<a href="https://colab.research.google.com/github/listenmusiceveryday/AIB6-TCM-Tongue-Classification/blob/main/convnext-small-328-model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Import อะไรต่างๆ เข้ามา


In [ ]:
# เช็คว่า GPU พร้อมใช้งานมั้ย
!nvidia-smi

In [ ]:
# โหลดแพ็คเกจเข้ามา
!pip install -q timm==0.9.12  # สำหรับ ConvNeXt
!pip install -q torch torchvision
!pip install -q pillow opencv-python-headless albumentations
!pip install -q scikit-learn pandas matplotlib seaborn
!pip install -q tensorboard tqdm

In [ ]:
# Import Library เข้ามา
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, OneCycleLR

import timm
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from tqdm.auto import tqdm
import warnings
import time
import albumentations as A
from albumentations.pytorch import ToTensorV2
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    multilabel_confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    accuracy_score
)

# ล็อคเส้นทางการสุ่มที่ 42
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"Timm: {timm.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# เคลียร์ขยะและคืนพื้นที่หน่วยความจำให้ GPU
import gc
import torch

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU RAM: {used:.2f} / {total:.2f} GB  ({(1-used/total)*100:.0f}% free)")

clear_memory()
print("Memory helpers ready")


## 2. Import Google drive, Setup Folder และกำหนดค่าต่างๆ

In [ ]:
# เชื่อมต่อกับ Google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/kaggle_dataset')
DATA_DIR     = PROJECT_ROOT / 'shezhenv3-txt'
IMAGE_DIR    = DATA_DIR / 'images'
LABELS_FILE  = DATA_DIR / 'tongue_dataset.csv'
OUTPUT_DIR   = PROJECT_ROOT / 'outputs'
MODEL_SAVE_DIR = OUTPUT_DIR / 'best_model_convnext_small'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLUMNS = [
    'Coating_White',
    'Coating_Yellow',
    'Texture_None',
    'Texture_Geographic',
    'Texture_Cracked',
    'Texture_Dentate'
]
NUM_LABELS = len(LABEL_COLUMNS)

CONVNEXT_MODEL = 'convnext_small' # ขนาดโมเดลเลือกได้ตั้งแต่ tiny, small, base, large
IMG_SIZE       = 328 # ขนาด input รูปภาพ
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✓ Project : {PROJECT_ROOT}")
print(f"✓ Images  : {IMAGE_DIR}")
print(f"✓ Labels  : {LABELS_FILE}")
print(f"✓ Classes : {NUM_LABELS}  →  {LABEL_COLUMNS}")
print(f"✓ Model   : {CONVNEXT_MODEL}")
print(f"✓ Device  : {DEVICE}")


## 3. Data Split หั่นออกมาในส่วน 80/10/10

In [ ]:
# โหลดไฟล์ CSV
df = pd.read_csv(LABELS_FILE)

In [ ]:
!pip install -q iterative-stratification
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

X = df[['Image_Name']].values
y = df[LABEL_COLUMNS].values

msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
train_val_idx, test_idx = next(msss1.split(X, y))

X_tv = X[train_val_idx]; y_tv = y[train_val_idx]
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.111, random_state=42)
train_idx, val_idx = next(msss2.split(X_tv, y_tv))

train_df = df.iloc[train_val_idx[train_idx]].reset_index(drop=True)
val_df   = df.iloc[train_val_idx[val_idx]].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

print(f"Train : {len(train_df):4d} ({len(train_df)/len(df)*100:.0f}%)")
print(f"Val   : {len(val_df):4d} ({len(val_df)/len(df)*100:.0f}%)")
print(f"Test  : {len(test_df):4d} ({len(test_df)/len(df)*100:.0f}%)")

print("\nLabel distribution check:")
for col in LABEL_COLUMNS:
    tr = train_df[col].mean(); vl = val_df[col].mean(); te = test_df[col].mean()
    print(f"  {col:25s}  train={tr:.2f}  val={vl:.2f}  test={te:.2f}")


## 4. Data Augmentation

In [ ]:
def get_train_transforms(img_size=328):
    return A.Compose([
        A.Resize(img_size, img_size),

        A.HorizontalFlip(p=0.5), # สุ่มโอกาสรูปที่ถูกพลิก ซ้าย-ขวา 50%

        # เอียง หมุน ซูม รูปภาพ
        A.ShiftScaleRotate(
            shift_limit=0.05, # สุ่มขยับภาพนิดหน่อย สุ่มขยับ ซ้าย, ขวา, บน, ล่าง 5%
            scale_limit=0.10, # ย่อ/ขยาย, zoom in หรือ zoom out ±10%
            rotate_limit=10,  # ทำการเอียงหรือหมุนรูปภาพซ้ายขวา ±10 องศา
            border_mode=0,    # ถมสีดำใส่ช่องว่างของรูปตอนเอียง ปกติถ้ารูปเอียงมันจะเกิด
            p=0.6             # โอกาศทั้งหมดในการสุ่มโดน 60%
        ),

        # จำลองสภาพแวดล้อม
        A.OneOf([                       # สุ่มเลือกทำอย่างใดอย่างนึงจาก 3 ฟังค์ชั่นนี้ โดยโอกาศทั้งหมดในการสุ่มโดน 80%
            A.RandomBrightnessContrast( # no.1 ความสว่าง & ความคมชัด
                brightness_limit=0.2,   # สุ่มปรับความสว่าง เพิ่มขึ้นหรือลดลง ±20%
                contrast_limit=0.2,     # สุ่มปรับความคมชัดของภาพ เพิ่มขึ้นหรือลดลง ±20%
                p=1.0                   # เมื่อถูกเลือกแล้ว บังคับทำ 100%
            ),
            A.HueSaturationValue(   # no.2 โทนสี & ความสดตัวของสี & ค่าความสว่าง
                hue_shift_limit=5,  # สุ่มเปลี่ยนโทนสีเล็กน้อย ใช้ค่า (hue) ไม่เกิน 5 หน่วย
                sat_shift_limit=20, # สุ่มปรับความสดของสี ใช้ค่า (saturation) ±20 หน่วย
                val_shift_limit=15, # สุ่มปรับค่าความสว่างของสี (value) ±15 หน่วย
                p=1.0               # เมื่อถูกเลือกแล้ว บังคับทำ 100%
            ),
            A.RandomGamma(              # no.3 สุ่มปรับค่าแกมม่า
                gamma_limit=(80, 120),  # สุ่มปรับค่าความเข้มของ Gradients ของแสง
                                        # หรือแสงในส่วนเงาและมิดโทน ในช่วง 80% ถึง 120%
                p=1.0                   # เมื่อถูกเลือกแล้ว บังคับทำ 100%
            ),
        ], p=0.8),

        # ใช้เทคนิค Contrast Limited Adaptive Histogram Equalization
        # เป็นเทคนิคขั้นสูงในการปรับความคมชัดและดึงรายละเอียดของภาพที่นิยมในภาพถ่ายการแพทย์
        A.CLAHE(
            clip_limit=2.0,         # ตัวคุม noise ทำให้ไฮท์ไลท์ได้ชัดขึ้น แต่คุมไว้ไม่ให้เร่งจนแตก
            tile_grid_size=(8, 8),  # แทนที่จะปรับแสงสว่างหรือมืดทั้งรูป CLAHE จะทำการ
                                    # หั่นรูปภาพออกเป็นขนาด 8x8 ช่อง (64ช่อง) แล้วไปปรับความคมชัดในช่อง
             p=0.3                  # โอกาศที่รูปภาพสุ่มโดน 30%
        ),

        # Blur & Noise จำลองกล้องมือถือที่คุณภาพต่างกัน โดยโอกาศสุ่มทั้งหมด 30%
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),   # จำลองกล้องหลุดโฟกัส เลนส์มัว
                                                        # ตั้งเบลอให้อยู่ระดับต่ำถึงปานกลาง ขนาดในการ blur จะสุ่มระหว่าง 3x3 ถึง 5x5 เบลอให้ขอบภาพนิ่มลง
                                                        # จำลองกรณีถ่ายแล้วลิ้นขยับตอนถ่ายพอดี หรือ กล้องสั่น หรือ มือถือมีคราบติดกล้อง
            A.GaussNoise(var_limit=(5.0, 20.0), p=1.0), # จำลองอาการภาพแตก
                                                        # ใส่ Noise กระจายบนภาพ และสุ่มค่าความหนาแน่นของ Noise ตั้งแต่ 5 ถึง 20
        ], p=0.3),

        # ปรับช่วงตัวเลขภาพให้อยู่มาตรฐานเดียวกันโมเดลที่เคยเรียนมาตอนแรกเพราะ ConvNeXT ได้ถูก
        # Pre-Trained มาจาก ImageNet ดังนั้นข้อมูลภาพชุดใหม่ของเราจึงต้องปรับโครงสร้าง
        # การกระจายตัวของสีให้ได้กับมาตรฐานเดียวกันกับโมเดลที่เคยเรียนมาตอนแรก
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
        ),
        ToTensorV2(), # แปลงรูปภาพกลายเป็น Tensor
    ])

def get_val_transforms(img_size=328):  # กำหนดขนาดรูปภาพ 328x328 pixel เฉพาะ Val กับ Test

    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
        ),
        ToTensorV2(), # แปลงรูปภาพกลายเป็น Tensor
    ])

print("Augmentation pipelines created")
print("Train: Geometric + Color + Blur augmentations")
print("Val/Test: Resize + Normalize only")

## 5. Create Dataset Class

In [ ]:
class TongueHealthDataset(Dataset):

    def __init__(self, df, image_dir, label_columns, transforms=None):
        self.df = df
        self.image_dir = Path(image_dir)
        self.label_columns = label_columns
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.image_dir / row['Image_Name']
        image = np.array(Image.open(img_path).convert('RGB'))

        labels = torch.tensor(
            [row[label] for label in self.label_columns],
            dtype=torch.float32
        )

        if self.transforms:
            transformed = self.transforms(image=image)
            image = transformed['image']

        return {
            'image': image,
            'labels': labels
        }

train_dataset = TongueHealthDataset(
    train_df, IMAGE_DIR, LABEL_COLUMNS, transforms=get_train_transforms(IMG_SIZE)
)
val_dataset = TongueHealthDataset(
    val_df, IMAGE_DIR, LABEL_COLUMNS, transforms=get_val_transforms(IMG_SIZE)
)
test_dataset = TongueHealthDataset(
    test_df, IMAGE_DIR, LABEL_COLUMNS, transforms=get_val_transforms(IMG_SIZE)
)

print(f"\n✓ Datasets created:")
print(f"  Train: {len(train_dataset)} (with augmentation)")
print(f"  Val:   {len(val_dataset)}")
print(f"  Test:  {len(test_dataset)}")

sample = train_dataset[0]
print(f"\nSample:")
print(f"  Image shape: {sample['image'].shape}")
print(f"  Labels shape: {sample['labels'].shape}")
print(f"  Labels: {sample['labels'].numpy()}")

## 6. Load ConvNeXT Model เข้ามาใน Notebook

In [ ]:
print(f"Loading ConvNeXt model: {CONVNEXT_MODEL}...")

class ConvNeXtClassifier(nn.Module):
    def __init__(self,
                 model_name='convnext_small',
                 num_labels=6,
                 pretrained=True,
                 dropout=0.3,
                 freeze_backbone=True
              ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0,
            global_pool='avg'
        )
        self.feature_dim = self.backbone.num_features

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
            print("  ✓ Backbone frozen (จะ unfreeze ที่ Phase 2)")

        self.classifier = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, num_labels)
        )
        self._init_classifier()

    def _init_classifier(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True
        print("  ✓ Backbone unfrozen (Phase 2 started)")

    def forward(self, x, labels=None):
        features  = self.backbone(x)
        logits    = self.classifier(features)
        loss      = None
        if labels is not None:
            loss = criterion(logits, labels)
        return {'loss': loss, 'logits': logits, 'features': features}


model = ConvNeXtClassifier(
    model_name=CONVNEXT_MODEL,
    num_labels=NUM_LABELS,
    pretrained=True,
    dropout=0.3,
    freeze_backbone=True,
).to(DEVICE)

def compute_pos_weight(train_df, label_columns, device):
    counts     = train_df[label_columns].sum()
    neg_counts = len(train_df) - counts
    pos_weight = (neg_counts / counts.clip(lower=1)).values
    print("\nPos weights (สูง = class หายาก):")
    for col, w in zip(label_columns, pos_weight):
        print(f"  {col:25s}: {w:.2f}")
    return torch.tensor(pos_weight, dtype=torch.float32).to(device)

pos_weight = compute_pos_weight(train_df, LABEL_COLUMNS, DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✓ Model ready  ({CONVNEXT_MODEL})")
print(f"  Total params    : {total_params:,}")
print(f"  Trainable params: {trainable_params:,}  (head only — backbone frozen)")


## 7. Training Configuration

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

BATCH_SIZE      = 16
NUM_EPOCHS      = 40
LR_HEAD         = 5e-4
LR_FINETUNE     = 2e-5
WEIGHT_DECAY    = 0.05
PATIENCE        = 8
UNFREEZE_EPOCH  = 5

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0,
                          pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999), eps=1e-8
)

from torch.optim.lr_scheduler import SequentialLR, LinearLR

warmup_scheduler = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=3)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCH - 3, eta_min=1e-5)
scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[3])

print("Training config (ConvNeXt-optimized):")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Max epochs     : {NUM_EPOCHS}")
print(f"  Phase 1 LR    : {LR_HEAD}  (epoch 1-{UNFREEZE_EPOCH}, head only)")
print(f"  Phase 2 LR    : {LR_FINETUNE}  (epoch {UNFREEZE_EPOCH+1}+, full model)")
print(f"  Weight decay   : {WEIGHT_DECAY}")
print(f"  Early stopping : patience={PATIENCE} (Phase 2 only)")
print(f"  Loss           : BCEWithLogitsLoss + pos_weight")

## 8. Training Function

In [ ]:
def compute_metrics(preds, labels, threshold=0.5):
    """
    Compute multi-label classification metrics
    """
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    probs = 1 / (1 + np.exp(-preds))
    y_pred = (probs > threshold).astype(int)
    y_true = labels.astype(int)

    metrics = {
        'exact_match': (y_pred == y_true).all(axis=1).mean(),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'recall_micro': recall_score(y_true, y_pred, average='micro', zero_division=0),
    }

    return metrics


def train_one_epoch(model, train_loader, optimizer, scheduler, device, epoch, scaler=None):
    """
    Train for one epoch with mixed precision
    """
    model.train()
    total_loss = 0

    running_preds  = []
    running_labels = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    for batch in pbar:
        images = batch['image'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        if scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(images, labels=labels)
                loss = outputs['loss']
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images, labels=labels)
            loss = outputs['loss']
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += loss.item()

        running_preds.append(outputs['logits'].detach().cpu().numpy())
        running_labels.append(labels.detach().cpu().numpy())

        del outputs, images, labels, loss
        torch.cuda.empty_cache()

        current_lr = scheduler.get_last_lr()[0]
        pbar.set_postfix({
            'loss': f"{total_loss / (pbar.n + 1):.4f}",
            'lr': f"{current_lr:.2e}"
        })

    avg_loss = total_loss / len(train_loader)
    all_preds  = np.concatenate(running_preds,  axis=0)
    all_labels = np.concatenate(running_labels, axis=0)
    metrics = compute_metrics(
        torch.tensor(all_preds), torch.tensor(all_labels)
    )
    metrics['loss'] = avg_loss
    del running_preds, running_labels, all_preds, all_labels

    return metrics


@torch.no_grad()
def evaluate(model, data_loader, device):
    """
    Evaluate model
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(data_loader, desc="Evaluating"):
        images = batch['image'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(images, labels=labels)

        total_loss += outputs['loss'].item()

        all_preds.append(outputs['logits'].cpu())
        all_labels.append(labels.cpu())
        del outputs, images, labels

    avg_loss = total_loss / len(data_loader)
    all_preds  = torch.cat(all_preds,  dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    metrics = compute_metrics(all_preds, all_labels)
    metrics['loss'] = avg_loss

    return metrics, all_preds, all_labels


print("Training functions defined")

## 9. Start training

In [ ]:
use_amp = True
scaler  = torch.cuda.amp.GradScaler() if use_amp else None

history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': [], 'lr': []}
best_val_f1      = 0
patience_counter = 0
phase2_started   = False
start_time       = time.time()

print("\n" + "=" * 70)
print(f"STARTING TRAINING — {CONVNEXT_MODEL.upper()}  (2-phase)")
print("=" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    if epoch == UNFREEZE_EPOCH + 1 and not phase2_started:
        model.unfreeze_backbone()
        phase2_started = True
        optimizer = AdamW([
            {'params': model.backbone.parameters(), 'lr': LR_FINETUNE},
            {'params': model.classifier.parameters(), 'lr': LR_FINETUNE * 10},
        ], weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999), eps=1e-8)

        from torch.optim.lr_scheduler import SequentialLR, LinearLR
        p2_warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=2)
        p2_cosine = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - UNFREEZE_EPOCH - 2, eta_min=1e-6)
        scheduler = SequentialLR(optimizer, schedulers=[p2_warmup, p2_cosine], milestones=[2])
        patience_counter = 0
        print(f"  → Phase 2 started at epoch {epoch}")

    clear_memory()
    train_metrics = train_one_epoch(model, train_loader, optimizer, scheduler, DEVICE, epoch, scaler)

    clear_memory()
    val_metrics, _, _ = evaluate(model, val_loader, DEVICE)

    history['train_loss'].append(train_metrics['loss'])
    history['train_f1'].append(train_metrics['f1_macro'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_f1'].append(val_metrics['f1_macro'])
    history['lr'].append(scheduler.get_last_lr()[0])

    if val_metrics['f1_macro'] > best_val_f1:
        best_val_f1 = val_metrics['f1_macro']
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_f1': best_val_f1,
            'label_columns': LABEL_COLUMNS,
            'model_name': CONVNEXT_MODEL,
            'img_size': IMG_SIZE,
        }, MODEL_SAVE_DIR / 'best_model.pt')
        print(f"  ✓ Best model saved  val F1={best_val_f1:.4f}")
    else:
        patience_counter += 1

    phase_tag = "P1-frozen" if epoch <= UNFREEZE_EPOCH else "P2-unfrozen"
    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} [{phase_tag}] "
          f"| train_loss={train_metrics['loss']:.4f} train_f1={train_metrics['f1_macro']:.4f} "
          f"| val_loss={val_metrics['loss']:.4f} val_f1={val_metrics['f1_macro']:.4f} "
          f"| lr={scheduler.get_last_lr()[0]:.2e} "
          f"| patience={patience_counter}/{PATIENCE}")

    if phase2_started and patience_counter >= PATIENCE:
        print(f"\n  Early stopping at epoch {epoch}")
        break

    scheduler.step()

total_time = time.time() - start_time
print(f"\n✓ Training done in {total_time/60:.1f} min   |   Best val F1: {best_val_f1:.4f}")

## 10. Plot Training History

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', marker='o', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', marker='s', linewidth=2)
axes[0].set_title('Loss', fontsize=14, weight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1 Score
axes[1].plot(history['train_f1'], label='Train', marker='o', linewidth=2)
axes[1].plot(history['val_f1'], label='Val', marker='s', linewidth=2)
axes[1].axhline(y=best_val_f1, color='r', linestyle='--', label=f'Best: {best_val_f1:.4f}')
axes[1].set_title('F1 Score (Macro)', fontsize=14, weight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Learning Rate
axes[2].plot(history['lr'], color='green', marker='o', linewidth=2)
axes[2].set_title('Learning Rate', fontsize=14, weight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('LR')
axes[2].set_yscale('log')
axes[2].grid(alpha=0.3)

plt.suptitle(f'Training History - {CONVNEXT_MODEL.upper()}', fontsize=16, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_history.png', dpi=300, bbox_inches='tight')
plt.show()

# Save history
with open(OUTPUT_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print("✓ Training history saved")

## 11. Load โมเดลที่ดีที่สุด แล้วนำมาประเมินผลโมเดลบน Test set

In [ ]:
# Load best model
checkpoint = torch.load(MODEL_SAVE_DIR / 'best_model.pt', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"✓ Best model loaded (Epoch {checkpoint['epoch']})")
print(f"  Validation F1: {checkpoint['best_val_f1']:.4f}")

# Evaluate on test set
print("\nEvaluating on test set...")
test_metrics, test_preds, test_labels = evaluate(model, test_loader, DEVICE)

print("\n" + "="*70)
print(f"TEST SET RESULTS - {CONVNEXT_MODEL.upper()}")
print("="*70)
print(f"Loss: {test_metrics['loss']:.4f}")
print(f"Exact Match Accuracy: {test_metrics['exact_match']:.4f}")
print(f"Hamming Loss: {test_metrics['hamming_loss']:.4f}")
print(f"\nF1 Scores:")
print(f"  Micro:    {test_metrics['f1_micro']:.4f}")
print(f"  Macro:    {test_metrics['f1_macro']:.4f}")
print(f"  Weighted: {test_metrics['f1_weighted']:.4f}")
print(f"\nPrecision (Micro): {test_metrics['precision_micro']:.4f}")
print(f"Recall (Micro):    {test_metrics['recall_micro']:.4f}")
print("="*70)

# Save test metrics
test_metrics['model_name'] = CONVNEXT_MODEL
test_metrics['img_size'] = IMG_SIZE
with open(OUTPUT_DIR / 'test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)

print(f"\n✓ Test metrics saved to {OUTPUT_DIR / 'test_metrics.json'}")

## 12. ดู score f1 แยกแต่ละ class

In [ ]:
# Per-class metrics
test_preds_np = test_preds.cpu().numpy()
test_labels_np = test_labels.cpu().numpy()

# Apply sigmoid and threshold
probs = 1 / (1 + np.exp(-test_preds_np))
y_pred = (probs > 0.5).astype(int)
y_true = test_labels_np.astype(int)

# Compute per-class metrics
per_class_f1 = {}
per_class_precision = {}
per_class_recall = {}

for i, label in enumerate(LABEL_COLUMNS):
    per_class_f1[label] = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
    per_class_precision[label] = precision_score(y_true[:, i], y_pred[:, i], zero_division=0)
    per_class_recall[label] = recall_score(y_true[:, i], y_pred[:, i], zero_division=0)

# Print per-class results
print("\n" + "="*70)
print("PER-CLASS PERFORMANCE")
print("="*70)
print(f"{'Class':<20} {'F1':>8} {'Precision':>12} {'Recall':>10} {'Support':>10}")
print("-" * 70)
for i, label in enumerate(LABEL_COLUMNS):
    support = y_true[:, i].sum()
    print(f"{label:<20} {per_class_f1[label]:>8.4f} {per_class_precision[label]:>12.4f} "
          f"{per_class_recall[label]:>10.4f} {support:>10.0f}")
print("="*70)

# Visualize per-class F1
plt.figure(figsize=(12, 6))
sorted_labels = sorted(per_class_f1.items(), key=lambda x: x[1], reverse=True)
labels, scores = zip(*sorted_labels)

colors = ['green' if s > 0.7 else 'orange' if s > 0.5 else 'red' for s in scores]
plt.barh(labels, scores, color=colors, alpha=0.7, edgecolor='black')
plt.xlabel('F1 Score', fontsize=12)
plt.title(f'Per-Class F1 Scores - {CONVNEXT_MODEL.upper()}', fontsize=14, weight='bold')
plt.xlim(0, 1)

for i, v in enumerate(scores):
    plt.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=10, weight='bold')

plt.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, linewidth=2)
plt.axvline(x=0.7, color='gray', linestyle='--', alpha=0.5, linewidth=2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'per_class_f1.png', dpi=300, bbox_inches='tight')
plt.show()

## 13. Confusion Matrices

In [ ]:
# Confusion matrix per label
cm_multi = multilabel_confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (label, cm) in enumerate(zip(LABEL_COLUMNS, cm_multi)):
    if i < len(axes):
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                    xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'],
                    cbar=False, annot_kws={'size': 12, 'weight': 'bold'})
        axes[i].set_title(f'{label}\nF1: {per_class_f1[label]:.3f}',
                         fontsize=11, weight='bold')
        axes[i].set_ylabel('True', fontsize=10)
        axes[i].set_xlabel('Predicted', fontsize=10)

# Hide extra subplot
if len(LABEL_COLUMNS) < len(axes):
    axes[-1].axis('off')

plt.suptitle(f'Confusion Matrices - {CONVNEXT_MODEL.upper()}',
             fontsize=14, weight='bold', y=1.00)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## 14. ลองดูตัวอย่างภาพที่โมเดลทายและเฉลยของรูปภาพนั้น

In [ ]:
# Visualize predictions on test images
def visualize_predictions(n_samples=8):
    fig, axes = plt.subplots(2, 4, figsize=(20, 11))
    axes = axes.flatten()

    indices = np.random.choice(len(test_df), n_samples, replace=False)

    for idx, ax in zip(indices, axes):
        row = test_df.iloc[idx]
        img_path = IMAGE_DIR / row['Image_Name']

        if img_path.exists():
            img = Image.open(img_path)
            ax.imshow(img)

            # Get predictions and ground truth
            true_labels = [LABEL_COLUMNS[i] for i in range(NUM_LABELS) if y_true[idx, i] == 1]
            pred_labels = [LABEL_COLUMNS[i] for i in range(NUM_LABELS) if y_pred[idx, i] == 1]

            # Check if prediction is correct
            is_correct = (y_true[idx] == y_pred[idx]).all()
            color = 'green' if is_correct else 'red'

            title = f"{'✓ CORRECT' if is_correct else '✗ WRONG'}\n\n"
            title += f"Ground Truth:\n{', '.join(true_labels) if true_labels else 'None'}\n\n"
            title += f"Predicted:\n{', '.join(pred_labels) if pred_labels else 'None'}"

            ax.set_title(title, fontsize=10, color=color, weight='bold')

        ax.axis('off')

    plt.suptitle(f'Sample Predictions - {CONVNEXT_MODEL.upper()}',
                 fontsize=16, weight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'sample_predictions.png', dpi=300, bbox_inches='tight')
    plt.show()

visualize_predictions()

## 15. เซฟโมเดลที่ดีที่สุดลงเครื่อง

In [ ]:
# Save detailed results
results_df = test_df.copy()

# Add predictions
for i, label in enumerate(LABEL_COLUMNS):
    results_df[f'pred_{label}'] = y_pred[:, i]
    results_df[f'prob_{label}'] = probs[:, i]

# Add correctness
results_df['correct'] = (y_true == y_pred).all(axis=1)

results_df.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

print(f"✓ Detailed results saved to {OUTPUT_DIR / 'test_predictions.csv'}")

# Save per-class metrics
per_class_metrics = {
    'f1': per_class_f1,
    'precision': per_class_precision,
    'recall': per_class_recall
}

with open(OUTPUT_DIR / 'per_class_metrics.json', 'w') as f:
    json.dump(per_class_metrics, f, indent=2)

print(f"✓ Per-class metrics saved to {OUTPUT_DIR / 'per_class_metrics.json'}")